# ASPECT parameter I/O examples

In [ ]:
import tempfile
from pathlib import Path

from gdmate.aspect_tools.io import parse_parameters_to_dict, save_parameters_from_dict

print("Imports successful.")

# Create a temporary directory for this notebook session
tmp_dir = Path(tempfile.mkdtemp())

print(f"Temporary working directory: {tmp_dir}")

## Parse a deal.ii / ASPECT parameter file into a nested dictionary

The function `parse_parameters_to_dict` reads a deal.ii / ASPECT-style parameter file and converts its hierarchical structure into a nested Python dictionary. It supports comments, `set key = value` entries, recursive `subsection ... end` blocks, repeated subsections (which are merged), and multi-line values continued with a trailing backslash (`\`). This makes it convenient for programmatically inspecting, modifying, or validating ASPECT parameter files after they are loaded into Python.

In [ ]:
# Locate test data file
repo_root = Path.cwd()
prm_path = (
    repo_root.parent.parent
    / "tests"
    / "data"
    / "aspect_tools"
    / "test_parse_parameters_to_dict.prm"
)

print(f"Using parameter file: {prm_path}")
assert prm_path.exists(), f"Test file not found: {prm_path}"

# Run parser
with open(prm_path, "r") as fin:
    params_dict = parse_parameters_to_dict(fin)

# Print parsed dictionary
print("\nParsed dictionary:")
print(params_dict)

## Write a nested dictionary to a deal.ii / ASPECT parameter file

The function `save_parameters_from_dict` takes a nested Python dictionary and serializes it into deal.ii / ASPECT parameter file syntax using `set`, `subsection`, and `end` blocks. String values are written as parameter entries, while nested dictionaries are written recursively as subsections with consistent indentation. The optional `indent_width` argument sets the number of spaces per subsection level and defaults to 4.

In [ ]:
# Parameters dictionary (same as pytest test)
parameters_dict = {
    'Dimension': '2',
    'Solver parameters': {
        'Stokes solver parameters': {
            'Linear solver tolerance': '1e-12'
        }
    },
    'Geometry model': {
        'Spherical shell': {
            'Inner radius': '1',
            'Outer radius': '2',
            'Opening angle': '360'
        }
    }
}

In [ ]:
# Write output to a temporary file in notebooks folder
# Indentation by 4 spaces
output_file_path = tmp_dir / "test_output_4s.prm"

with open(output_file_path, "w") as fout:
    save_parameters_from_dict(fout, parameters_dict)

print(f"\nGenerated output file: {output_file_path}")
print("\nFile content:\n")
print(output_file_path.read_text())

In [ ]:
# Write output to a temporary file in notebooks folder
# Indentation by 2 spaces
output_file_path = tmp_dir / "test_output_2s.prm"

with open(output_file_path, "w") as fout:
    save_parameters_from_dict(fout, parameters_dict, indent_width=2)

print(f"\nGenerated output file: {output_file_path}")
print("\nFile content:\n")
print(output_file_path.read_text())

## Modify an ASPECT cookbook parameter file

This example reads the three-dimensional convection-box cookbook with `parse_parameters_to_dict`, changes both initial mesh-refinement levels from 3 to 4, and writes the modified model to a new parameter file with `save_parameters_from_dict`.

In [ ]:
cookbook_prm_path = (
    repo_root.parent.parent
    / "tests"
    / "data"
    / "aspect_tools"
    / "convection_box_3d.prm"
)

with cookbook_prm_path.open() as fin:
    convection_box_parameters = parse_parameters_to_dict(fin)

mesh_refinement = convection_box_parameters["Mesh refinement"]
mesh_refinement["Initial global refinement"] = "4"
mesh_refinement["Initial adaptive refinement"] = "4"

modified_prm_path = tmp_dir / "convection_box_3d_refinement_4_4.prm"
with modified_prm_path.open("w") as fout:
    save_parameters_from_dict(fout, convection_box_parameters)

print(f"Read parameter file: {cookbook_prm_path}")
print("Set initial global and adaptive refinement levels to 4.")
print(f"Wrote modified parameter file: {modified_prm_path}")

## Alternative: include the base parameter file

ASPECT's `include` directive can keep the modified parameter file small by including a previous file. We also have the option to pass `include_path` to `save_parameters_from_dict`, which writes the corresponding `include` directive before the parameters in the dictionary. The included file is read at the position of the directive, and parameters set afterward override values from that file. Relative include paths are interpreted from the directory where ASPECT is run, so this example writes the resolved absolute path of the test fixture.

In [ ]:
refinement_overrides = {
    "Mesh refinement": {
        "Initial global refinement": "4",
        "Initial adaptive refinement": "4",
    }
}
include_prm_path = tmp_dir / "convection_box_3d_include_refinement_4_4.prm"

with include_prm_path.open("w") as fout:
    save_parameters_from_dict(
        fout,
        refinement_overrides,
        include_path=cookbook_prm_path.resolve(),
    )

print(f"Wrote include-based parameter file: {include_prm_path}")
print("\nFile content:\n")
print(include_prm_path.read_text())